# Cell Translator — morphology perturbation: QC-reviewed end-to-end rerun

this notebook preserves its PyTorch morphology autoencoder and TensorFlow CycleGAN definitions, with a **70/15/15 train/validation/test** division. All 10 translations are saved as original-scale morphology arrays plus ordered features.

**Methodological constraints:** CycleGANs retain the original fixed-epoch training approach; their validation sets are reserved for future hyperparameter selection and are never used as final test sets. Do not change the configuration after inspecting test metrics. separate structural QC, training-only feature selection and imputation, cell-level QC audits, validation-only AE early stopping, untouched test evaluation. Prior outputs are preserved under a new run directory.


In [ ]:
import os, re, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path(os.environ.get("CELL_TRANSLATOR_DATA_DIR", str(Path.cwd() / "data"))).expanduser().resolve()
RUN_ROOT = Path(os.environ.get("CELL_TRANSLATOR_RUN_DIR", str(Path.cwd() / "outputs" / "run_70_15_15_qc_v3"))).expanduser().resolve()
AE_DIR = str(RUN_ROOT / "autoencoders")
DEV_DIR = str(RUN_ROOT / "cyclegan_ipsc_astrocyte_drugs")
CTRL_DIR = str(RUN_ROOT / "cyclegan_control_to_perturbation")
EVAL_DIR = str(RUN_ROOT / "evaluation")
for path in [AE_DIR, DEV_DIR, CTRL_DIR, EVAL_DIR]:os.makedirs(path,exist_ok=True)
SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

MIN_NONMISSING_FRACTION = 0.95 
MIN_STD = 1e-8
# Cell-level missingness: flag only, never automatically remove a cell, review then remove.
CELL_MISSINGNESS_FLAG_THRESHOLD = 0.20
# Standardized extreme values: report, do not clip/remove automatically, review the clip.
EXTREME_Z_THRESHOLD = 10.0
CONDITIONS = [f"{t}_{d}" for t in ("iPSC","Astrocytes") for d in ("DMSO","Latrunculin_B","GW3965","Clozapine")]
DRUGS = ["Latrunculin_B","GW3965","Clozapine"]
ALL_CONDITIONS = ["DMSO"] + DRUGS
np.random.seed(SEED);random.seed(SEED);torch.manual_seed(SEED)
print("Run root:",RUN_ROOT)
assert BASE_DIR.exists(), f"Missing morphology input data directory: {BASE_DIR}"


## 1. Input audit and immutable metadata checks (before splitting)

Use the same cleaned CellProfiler morphology CSVs as in the original analysis.


In [ ]:
GROUP_COLUMN = None  
raw = {}
for name in CONDITIONS:
    file = BASE_DIR / f"{name}.csv"
    if not file.exists(): raise FileNotFoundError(file)
    raw[name] = pd.read_csv(file)
    print(f"{name}: {raw[name].shape}")
if GROUP_COLUMN:
    assert all(GROUP_COLUMN in df.columns for df in raw.values()), "Missing requested group column"


In [ ]:
def is_forbidden_position_feature(feature):

    f = str(feature)

    patterns = [
        r"AreaShape_BoundingBox(?:Minimum|Maximum)_[XYZ]$",
        r"AreaShape_(?:Center|Centroid)_[XYZ]$",
        r"(^|_)Location_",
        r"(^|_)ImageNumber($|_)",
        r"(^|_)ObjectNumber($|_)",
        r"^Metadata_",
    ]

    return any(
        re.search(
            pattern,
            f,
            flags=re.IGNORECASE,
        )
        is not None

        for pattern in patterns
    )




In [ ]:
def numeric_features(df):
    """Extract eligible morphology measurements; do not use full-data statistics to select them."""
    if df.columns.duplicated().any():
        dup = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate column names detected: {dup[:20]}")
    excluded = [c for c in df.columns if is_forbidden_position_feature(c)]
    cols = [c for c in df.columns if c not in excluded and c != GROUP_COLUMN and not str(c).startswith("Unnamed:")]
    # Keep all candidate numeric columns, including entirely missing ones: TRAIN-only QC will drop them.
    X = df[cols].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf],np.nan)
    if X.shape[1] == 0:
        raise ValueError("No candidate morphology columns remain after metadata exclusions")
    return X

def audit_morphology_data(name, df):
    X = numeric_features(df)
    row_missing = X.isna().mean(axis=1)
    report = pd.DataFrame({
        "row_id": np.arange(len(X), dtype=int),
        "missing_fraction": row_missing.to_numpy(),
        "flag_high_missingness": (row_missing > CELL_MISSINGNESS_FLAG_THRESHOLD).to_numpy(),
    })
    prefix = os.path.join(AE_DIR, name)
    report.to_csv(prefix + "_cell_qc.csv", index=False)
    summary = {
        "condition": name,
        "n_cells": len(X), "n_candidate_features": X.shape[1],
        "n_missingness_flagged": int(report.flag_high_missingness.sum()),
        "fraction_missing_values": float(X.isna().to_numpy().mean()),
        "n_all_missing_features_audit_only": int(X.isna().all(axis=0).sum()),
    }
    return summary

audit_summary = pd.DataFrame([audit_morphology_data(name, df) for name, df in raw.items()])
audit_summary.to_csv(os.path.join(AE_DIR, "input_qc_summary.csv"), index=False)
display(audit_summary)

## 2. Freeze 70/15/15 partitions, then fit preprocessing on **training cells only**

In [ ]:
def create_split(n, groups=None, seed=SEED):
    idx = np.arange(n)
    if groups is None:
        tr, rem = train_test_split(idx, train_size=TRAIN_FRAC, random_state=seed, shuffle=True)
        va, te = train_test_split(rem, train_size=VAL_FRAC/(VAL_FRAC+TEST_FRAC), random_state=seed, shuffle=True)
    else:
        groups = np.asarray(groups).astype(str)
        if len(groups) != n:
            raise ValueError("Group labels must match number of rows")
        if len(np.unique(groups)) < 3:
            raise ValueError("Group-aware three-way split requires at least three distinct groups")
        gss = GroupShuffleSplit(n_splits=1, train_size=TRAIN_FRAC, random_state=seed)
        a, b = next(gss.split(idx, groups=groups)); tr, rem = idx[a], idx[b]
        gss2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=seed + 1)
        a, b = next(gss2.split(rem, groups=groups[rem])); va, te = rem[a], rem[b]
        for left, right in ((tr,va),(tr,te),(va,te)):
            assert not set(groups[left]).intersection(groups[right]), "Group leakage in split"
    assert len(np.unique(np.r_[tr, va, te])) == n, "Split row duplication or omissions"
    return {"train": np.sort(tr), "val": np.sort(va), "test": np.sort(te)}

def prepare_ae_condition(name, df):
    X = numeric_features(df)
    idx = create_split(len(X), df[GROUP_COLUMN].to_numpy() if GROUP_COLUMN else None)
    tr = X.iloc[idx["train"]]
    completeness = tr.notna().mean(axis=0)
    nonzero = tr.fillna(0).ne(0).mean(axis=0)  # audit ONLY; zeros can be meaningful
    train_std = tr.std(axis=0, ddof=1)
    keep = ((completeness >= MIN_NONMISSING_FRACTION)
            & np.isfinite(train_std)
            & (train_std > MIN_STD))
    features = keep.index[keep].tolist()
    if not features:
        raise ValueError(f"{name}: no features survive train-only missingness and variance QC")
    if any(is_forbidden_position_feature(f) for f in features):
        raise AssertionError(f"Forbidden positional feature reached autoencoder in {name}")
    med = tr[features].median(axis=0)
    if med.isna().any():
        raise ValueError(f"{name}: selected features have undefined training medians")
    scaler = StandardScaler().fit(tr[features].fillna(med))
    prefix = os.path.join(AE_DIR, name)
    audit = pd.DataFrame({
        "feature": X.columns.to_numpy(),
        "nonmissing_fraction_train": completeness.to_numpy(),
        "nonzero_fraction_train_audit_only": nonzero.to_numpy(),
        "std_train": train_std.to_numpy(),
        "keep": keep.to_numpy(),
    })
    audit["drop_reason"] = np.select(
        [audit["nonmissing_fraction_train"] < MIN_NONMISSING_FRACTION,
         ~np.isfinite(audit["std_train"]) | (audit["std_train"] <= MIN_STD)],
        ["low_train_completeness", "low_or_nonfinite_train_variance"],
        default="retained",
    )
    audit.to_csv(prefix + "_feature_qc.csv", index=False)
    pd.DataFrame({
        "feature": features,
        "train_median": med.to_numpy(),
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
    }).to_csv(prefix + "_scaler_stats.csv", index=False)
    assignment = np.full(len(X), "unused", dtype=object)
    extreme_rows = []
    for split, rows in idx.items():
        assignment[rows] = split
        # Fill missing values with training medians, and use the TRAIN-fitted scaler.
        matrix = scaler.transform(X.iloc[rows][features].fillna(med)).astype(np.float32)
        if not np.isfinite(matrix).all():
            raise ValueError(f"{name}/{split}: non-finite values after preprocessing")
        np.save(prefix + f"_X_{split}_scaled.npy", matrix)
        np.save(prefix + f"_{split}_idx.npy", rows)
        extreme_rows.append(pd.DataFrame({
            "feature": features, "split": split,
            "fraction_abs_z_gt_threshold": (np.abs(matrix) > EXTREME_Z_THRESHOLD).mean(axis=0),
            "max_abs_z": np.abs(matrix).max(axis=0),
        }))
    pd.concat(extreme_rows, ignore_index=True).to_csv(prefix + "_extreme_z_qc.csv", index=False)
    assignments = pd.DataFrame({
        "row_id": np.arange(len(X)),
        "split": assignment,
        "group": df[GROUP_COLUMN].astype(str) if GROUP_COLUMN else "",
    })
    assignments.to_csv(prefix + "_split_assignments.csv", index=False)
    print(f"{name}: {len(features)}/{X.shape[1]} features retained; "
          f"splits { {k: len(v) for k,v in idx.items()} }")
    return {"name": name, "feature_cols": features, "splits": idx,
            "train_median": med, "scaler": scaler, "feature_qc": audit}

if GROUP_COLUMN:
    print("WARNING: per-condition group splits may assign the same donor/group to different splits "
          "across conditions. For true cross-condition held-out donor evaluation, build a shared "
          "group-to-split assignment before running this cell.")

prepared = {name: prepare_ae_condition(name, raw[name]) for name in CONDITIONS}
with open(RUN_ROOT / "run_config.json", "w") as f:
    json.dump({"seed": SEED, "split": [TRAIN_FRAC, VAL_FRAC, TEST_FRAC],
               "group_column": GROUP_COLUMN, "min_nonmissing_fraction": MIN_NONMISSING_FRACTION,
               "min_std": MIN_STD, "cell_missingness_flag_threshold": CELL_MISSINGNESS_FLAG_THRESHOLD,
               "extreme_z_threshold": EXTREME_Z_THRESHOLD,
               "conditions": CONDITIONS, "source_notebook": "Morph_perturbation.ipynb"}, f, indent=2)

## 3. Train 8 autoencoders; validation only for early stopping


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import matplotlib.pyplot as plt



LATENT_DIM = 128     
NUM_EPOCHS = 100
BATCH_SIZE = 128
LR = 1e-4
PATIENCE = 20

CONDITIONS = [
    "iPSC_DMSO", "iPSC_Clozapine", "iPSC_GW3965", "iPSC_Latrunculin_B",
    "Astrocytes_DMSO", "Astrocytes_Clozapine", "Astrocytes_GW3965", "Astrocytes_Latrunculin_B",
]


class NumpyDataset(Dataset):
    def __init__(self, arr):
        self.data = torch.tensor(arr, dtype=torch.float32)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, i):
        return self.data[i]


class MorphologyAutoencoder(nn.Module):
    """
    Standard AE with BatchNorm + LeakyReLU. Linear output layer for
    standardized continuous morphology features — no Hurdle head, since
    morphology values are dense/continuous (unlike scRNA-seq dropout data).
    """
    def __init__(self, input_dim, latent_dim=128):
        super().__init__()

        def enc_block(d_in, d_out):
            return nn.Sequential(
                nn.Linear(d_in, d_out), nn.BatchNorm1d(d_out),
                nn.LeakyReLU(0.1), nn.Dropout(0.15))

        def dec_block(d_in, d_out, drop=True):
            layers = [nn.Linear(d_in, d_out), nn.BatchNorm1d(d_out), nn.LeakyReLU(0.1)]
            if drop:
                layers.append(nn.Dropout(0.15))
            return nn.Sequential(*layers)

        self.encoder = nn.Sequential(
            enc_block(input_dim, 1024),
            enc_block(1024, 512),
            nn.Sequential(nn.Linear(512, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.1)),
            nn.Linear(256, latent_dim),
        )
        self.decoder = nn.Sequential(
            dec_block(latent_dim, 256),
            dec_block(256, 512),
            dec_block(512, 1024, drop=False),
            nn.Linear(1024, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return z, recon

    def encode(self, x):
        return self.encoder(x)


def morphology_loss(recon, target, lambda_var=0.5, lambda_cov=0.1):
    mse = F.mse_loss(recon, target)
    std_real = target.std(dim=0) + 1e-8
    std_recon = recon.std(dim=0) + 1e-8
    var_loss = F.mse_loss(std_recon, std_real)
    recon_c = recon - recon.mean(dim=0, keepdim=True)
    cov = (recon_c.T @ recon_c) / (recon.shape[0] - 1)
    off_diag = cov - torch.diag(torch.diag(cov))
    cov_loss = (off_diag ** 2).mean()
    return mse + lambda_var * var_loss + lambda_cov * cov_loss


def train_morphology_ae(model, train_dl, val_dataset, device,
                         num_epochs=150, lr=1e-4, patience=20,
                         lambda_var=0.5, lambda_cov=0.1):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=8)

    best_val, best_state, best_ep, wait = float("inf"), None, -1, 0
    train_losses, val_losses, latent_vars = [], [], []
    val_data = val_dataset.data.to(device)

    for ep in range(1, num_epochs + 1):
        model.train()
        t = 0.0
        for batch in train_dl:
            batch = batch.to(device)
            _, recon = model(batch)
            loss = morphology_loss(recon, batch, lambda_var=lambda_var, lambda_cov=lambda_cov)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            t += float(loss)

        train_loss = t / max(len(train_dl), 1)

        model.eval()
        with torch.no_grad():
            z_val, recon_val = model(val_data)
            val_loss = float(F.mse_loss(recon_val, val_data))
            latent_var = float(z_val.var(dim=0).mean())

        scheduler.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        latent_vars.append(latent_var)

        print(f"Epoch {ep:03d}/{num_epochs} | train={train_loss:.5f} | val={val_loss:.5f} | "
              f"latent_var={latent_var:.5f} | lr={optimizer.param_groups[0]['lr']:.2e}")

        if val_loss < best_val - 1e-6:
            best_val, best_ep, wait = val_loss, ep, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at {ep}. Best {best_ep} | val={best_val:.5f}")
                break

    if best_state:
        model.load_state_dict(best_state)
    return {"train_losses": train_losses, "val_losses": val_losses,
            "latent_vars": latent_vars, "best_epoch": best_ep, "best_val": best_val}


def get_latents(model, X, device, batch_size=256):
    dl = DataLoader(NumpyDataset(X), batch_size=batch_size, shuffle=False)
    Zs = []
    model.eval()
    with torch.no_grad():
        for batch in dl:
            Zs.append(model.encode(batch.to(device)).cpu().numpy())
    return np.concatenate(Zs, 0)


def get_reconstructions(model, X, device, batch_size=256):
    dl = DataLoader(NumpyDataset(X), batch_size=batch_size, shuffle=False)
    Rs = []
    model.eval()
    with torch.no_grad():
        for batch in dl:
            _, recon = model(batch.to(device))
            Rs.append(recon.cpu().numpy())
    return np.concatenate(Rs, 0)


def compute_reconstruction_metrics(X_real, X_recon, scaler=None, name=""):
    if scaler is not None:
        X_r = scaler.inverse_transform(X_real)
        X_h = scaler.inverse_transform(X_recon)
    else:
        X_r, X_h = X_real, X_recon

    mse_cells = ((X_recon - X_real) ** 2).mean(axis=1)
    mse_features = ((X_recon - X_real) ** 2).mean(axis=0)

    ss_res = ((X_r - X_h) ** 2).sum(axis=0)
    ss_tot = ((X_r - X_r.mean(axis=0, keepdims=True)) ** 2).sum(axis=0)
    r2_per_feature = 1.0 - ss_res / np.maximum(ss_tot, 1e-12)
    median_r2 = float(np.median(r2_per_feature))
    frac_good = float((r2_per_feature > 0.5).mean())

    r_feature, _ = pearsonr(X_r.mean(0), X_h.mean(0))
    r_std, _ = pearsonr(X_r.std(0), X_h.std(0))

    n_sample = min(X_r.shape[0], 5000)
    rng = np.random.default_rng(0)
    idx = rng.choice(X_r.shape[0], n_sample, replace=False)
    r_overall, _ = pearsonr(X_r[idx].ravel(), X_h[idx].ravel())

    print(f"\n[{name}] Reconstruction metrics (original scale):")
    print(f"  Median R² per feature:    {median_r2:.4f}  (target > 0.50)")
    print(f"  Fraction features R²>0.5: {frac_good:.4f}  (target > 0.50)")
    print(f"  Feature-mean Pearson r:   {r_feature:.4f}  (target > 0.90)")
    print(f"  Feature-std  Pearson r:   {r_std:.4f}      (target > 0.85)")
    print(f"  Overall value Pearson r:  {r_overall:.4f}  (target > 0.90)")

    return {"mse_cells": mse_cells, "mse_features": mse_features,
            "r2_per_feature": r2_per_feature, "median_r2": median_r2,
            "frac_good_r2": frac_good, "r_feature": float(r_feature),
            "r_std": float(r_std), "r_overall": float(r_overall)}


def plot_loss_curves(hist, name, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=300)
    epochs = np.arange(1, len(hist["train_losses"]) + 1)
    ax = axes[0]
    ax.plot(epochs, hist["train_losses"], label="Train", lw=2)
    ax.plot(epochs, hist["val_losses"], label="Val", lw=2)
    ax.axvline(hist["best_epoch"], color="black", lw=1.2, ls="--",
               label=f"Best epoch {hist['best_epoch']}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
    ax.set_title(f"{name} — Loss curves"); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.plot(epochs, hist["latent_vars"], color="darkorange", lw=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Mean latent variance")
    ax.set_title(f"{name} — Latent variance"); ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_reconstruction_quality(X_real, X_recon, metrics, name, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=300)

    ax = axes[0]
    ax.hist(metrics["mse_cells"], bins=60, alpha=0.8, color="#4C72B0")
    ax.axvline(np.median(metrics["mse_cells"]), color="red", lw=1.5, ls="--",
               label=f"median={np.median(metrics['mse_cells']):.4f}")
    ax.set_xlabel("Per-cell MSE"); ax.set_ylabel("Count")
    ax.set_title(f"{name} — Per-cell reconstruction error")
    ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    x, y = X_real.mean(0), X_recon.mean(0)
    ax.scatter(x, y, s=4, alpha=0.4)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.2)
    ax.set_xlabel("Real feature mean"); ax.set_ylabel("Reconstructed feature mean")
    ax.set_title(f"{name} — Feature mean: r={metrics['r_feature']:.3f}")
    ax.grid(alpha=0.25)

    ax = axes[2]
    x, y = X_real.std(0), X_recon.std(0)
    ax.scatter(x, y, s=4, alpha=0.4, color="#DD8452")
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.2)
    ax.set_xlabel("Real feature std"); ax.set_ylabel("Reconstructed feature std")
    ax.set_title(f"{name} — Feature std: r={metrics['r_std']:.3f}")
    ax.grid(alpha=0.25)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()




In [ ]:
LATENT_DIM=128; NUM_EPOCHS=100; BATCH_SIZE=128; LR=1e-4; PATIENCE=20

def fit_autoencoders():
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    summary=[]
    for name in CONDITIONS:
        prefix=os.path.join(AE_DIR,name)
        stats=pd.read_csv(prefix+'_scaler_stats.csv')
        Xtr=np.load(prefix+'_X_train_scaled.npy')
        Xva=np.load(prefix+'_X_val_scaled.npy')
        Xte=np.load(prefix+'_X_test_scaled.npy')
        model=MorphologyAutoencoder(input_dim=Xtr.shape[1],latent_dim=LATENT_DIM).to(device)
        train_dl=DataLoader(NumpyDataset(Xtr),batch_size=BATCH_SIZE,shuffle=True,drop_last=False)
        history=train_morphology_ae(model,train_dl,NumpyDataset(Xva),device,num_epochs=NUM_EPOCHS,lr=LR,patience=PATIENCE,lambda_var=2.0,lambda_cov=0.1)
        # Original trainer restores best validation state before returning.
        torch.save({'model_state_dict':model.state_dict(),'config':{'input_dim':Xtr.shape[1],'latent_dim':LATENT_DIM,'num_epochs':NUM_EPOCHS,'lr':LR},'history':history,'feature_cols':stats.feature.tolist(), **{f'{split}_idx':np.load(prefix+f'_{split}_idx.npy') for split in ('train','val','test')}}, prefix+'_ae_checkpoint.pt')
        for split, X in [('train',Xtr),('val',Xva),('test',Xte)]:
            np.save(prefix+f'_Z_{split}.npy',get_latents(model,X,device))
        recon=get_reconstructions(model,Xte,device)
        scores=compute_reconstruction_metrics(Xte,recon,scaler=None,name=name+'_test')
        summary.append({'condition':name,'n_train':len(Xtr),'n_val':len(Xva),'n_test':len(Xte),'n_features':Xtr.shape[1],'best_epoch':history['best_epoch'],'test_median_r2_scaled':scores['median_r2'],'test_feature_mean_r_scaled':scores['r_feature']})
        pd.DataFrame({'epoch':np.arange(1,len(history['train_losses'])+1),'train_loss':history['train_losses'],'validation_loss':history['val_losses']}).to_csv(prefix+'_training_history.csv',index=False)
    out=pd.DataFrame(summary);out.to_csv(os.path.join(AE_DIR,'ae_test_summary.csv'),index=False)
    return out

ae_summary=fit_autoencoders()
display(ae_summary)


## 4. Train four developmental translators (iPSC → Astrocytes)


In [ ]:
"""
iPSC_drug -> Astrocytes_drug CycleGAN translation
"""

import os
import numpy as np
import pandas as pd
import torch
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input, Dense, LayerNormalization
from tensorflow.keras.optimizers import Adam
from scipy.stats import pearsonr, ks_2samp, wasserstein_distance
import matplotlib.pyplot as plt
import umap

AE_DIR = AE_DIR
OUT_ROOT = DEV_DIR
os.makedirs(OUT_ROOT, exist_ok=True)

DRUG_SLUGS = {
    "DMSO": "DMSO",
    "Latrunculin_B": "Latrunculin_B",
    "GW3965": "GW3965",
    "Clozapine": "Clozapine",
}

BATCH_SIZE = 32
EPOCHS = 300
GP_WEIGHT = 10.0
DRIFT_EPS = 1e-3
N_CRITIC = 5
LAMBDA_CYCLE = 5.0
LAMBDA_IDENTITY = 0.5
PAIR_W, MMD_W, COV_W, MOM_W = 1.0, 1.0, 0.2, 2.0

def load_ae_outputs(name, out_dir):
    prefix = os.path.join(out_dir, name)
    scaler_stats = pd.read_csv(prefix + "_scaler_stats.csv")
    return {
        "name": name,
        "Z_train": np.load(prefix + "_Z_train.npy"),
        "Z_test": np.load(prefix + "_Z_test.npy"),
        "Z_val": np.load(prefix + "_Z_val.npy"),
        "X_train": np.load(prefix + "_X_train_scaled.npy"),
        "X_test": np.load(prefix + "_X_test_scaled.npy"),
        "X_val": np.load(prefix + "_X_val_scaled.npy"),
        "tr_idx": np.load(prefix + "_train_idx.npy"),
        "val_idx": np.load(prefix + "_val_idx.npy"),
        "te_idx": np.load(prefix + "_test_idx.npy"),
        "scaler_stats": scaler_stats,
        "feature_cols": scaler_stats["feature"].tolist(),
    }


def std_z(Z, Z_train_ref):
    mean = Z_train_ref.mean(0, keepdims=True)
    std = np.maximum(Z_train_ref.std(0, keepdims=True), 1e-6)
    return (Z - mean) / std, mean, std


def build_generator(d_in, d_out, name):
    x = inp = Input(shape=(d_in,))
    x = LayerNormalization()(Dense(512, activation="relu")(x))
    x = LayerNormalization()(Dense(256, activation="relu")(x))
    x = LayerNormalization()(Dense(256, activation="relu")(x))
    x = Dense(d_out)(x)
    return models.Model(inp, x, name=name)


def build_discriminator(d, name):
    x = inp = Input(shape=(d,))
    x = Dense(256, activation="relu")(x)
    x = Dense(128, activation="relu")(x)
    x = Dense(64, activation="relu")(x)
    x = Dense(1)(x)
    return models.Model(inp, x, name=name)


def generator_loss(fake_scores):
    return -tf.reduce_mean(fake_scores)


def discriminator_loss(r, f):
    return tf.reduce_mean(f) - tf.reduce_mean(r)


def cycle_loss(a, b):
    return tf.reduce_mean(tf.abs(a - b))


def identity_loss(a, b):
    return tf.reduce_mean(tf.abs(a - b))


def gradient_penalty(disc, real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1])
    inter = real + alpha * (fake - real)
    with tf.GradientTape() as t:
        t.watch(inter)
        pred = disc(inter)
    grad = t.gradient(pred, inter)
    norm = tf.sqrt(tf.reduce_sum(tf.square(grad), axis=1) + 1e-12)
    return tf.reduce_mean((norm - 1.0) ** 2)


def knn_pair_loss_cos(fake, bank, k=5, bank_sample=8192):
    n = tf.shape(bank)[0]
    m = tf.minimum(bank_sample, n)
    idx = tf.random.uniform([m], 0, n, dtype=tf.int32)
    B = tf.gather(bank, idx)
    f = tf.math.l2_normalize(fake, axis=1)
    b = tf.math.l2_normalize(B, axis=1)
    sims = tf.matmul(f, b, transpose_b=True)
    _, inds = tf.math.top_k(sims, k=k)
    b_knn = tf.gather(b, inds, axis=0)
    return tf.reduce_mean(1.0 - tf.reduce_sum(f * tf.reduce_mean(b_knn, axis=1), axis=1))


def mmd_rbf(x, y, sigmas=(0.5, 1.0, 2.0)):
    def k(a, b, s):
        a2 = tf.reduce_sum(a * a, 1, keepdims=True)
        b2 = tf.reduce_sum(b * b, 1, keepdims=True)
        d2 = a2 - 2.0 * tf.matmul(a, b, transpose_b=True) + tf.transpose(b2)
        d2 = tf.maximum(d2, 0.0)
        return tf.exp(-d2 / (2 * s * s))

    kxx = tf.add_n([k(x, x, s) for s in sigmas]) / len(sigmas)
    kyy = tf.add_n([k(y, y, s) for s in sigmas]) / len(sigmas)
    kxy = tf.add_n([k(x, y, s) for s in sigmas]) / len(sigmas)
    return tf.reduce_mean(kxx) + tf.reduce_mean(kyy) - 2.0 * tf.reduce_mean(kxy)


def moment_loss(fake, real):
    mean_term = tf.reduce_mean(
        tf.abs(tf.reduce_mean(fake, 0) - tf.reduce_mean(real, 0))
    )
    std_term = tf.reduce_mean(
        tf.abs(tf.math.reduce_std(fake, 0) - tf.math.reduce_std(real, 0))
    )
    return mean_term + std_term


def cov_loss(fake, real):
    def cov(x):
        xc = x - tf.reduce_mean(x, 0, keepdims=True)
        return tf.matmul(xc, xc, transpose_a=True) / tf.cast(tf.shape(xc)[0] - 1, tf.float32)

    return tf.reduce_mean(tf.square(cov(fake) - cov(real)))

def run_drug_translation(drug_display_name, ae_dir=AE_DIR, out_root=OUT_ROOT,
                          epochs=EPOCHS, batch_size=BATCH_SIZE, save_plots=True):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)
    np.random.seed(42)

    slug = DRUG_SLUGS[drug_display_name]
    SRC_NAME = f"iPSC_{slug}"
    TGT_NAME = f"Astrocytes_{slug}"

    drug_out_dir = os.path.join(out_root, slug.replace(" ", "_"))
    os.makedirs(drug_out_dir, exist_ok=True)
    print(f"\n{'=' * 70}\n{drug_display_name}  ({SRC_NAME} -> {TGT_NAME})\n{'=' * 70}")

    # ---- Load AE outputs ----------------------------------------------------
    src_ae = load_ae_outputs(SRC_NAME, ae_dir)
    tgt_ae = load_ae_outputs(TGT_NAME, ae_dir)

    src_feature_cols = src_ae["feature_cols"]
    tgt_feature_cols = tgt_ae["feature_cols"]
    if src_feature_cols != tgt_feature_cols:
        n_common = len(set(src_feature_cols) & set(tgt_feature_cols))
        print(f"[info] {SRC_NAME} has {len(src_feature_cols)} features, "
              f"{TGT_NAME} has {len(tgt_feature_cols)} features "
              f"({n_common} in common). This is expected — decoding uses "
              f"each cell type's own decoder/feature space.")

    # ---- Standardize latents using train stats -------------------------------
    train_src_z, mean_src, std_src = std_z(src_ae["Z_train"], src_ae["Z_train"])
    train_tgt_z, mean_tgt, std_tgt = std_z(tgt_ae["Z_train"], tgt_ae["Z_train"])
    test_src_z = (src_ae["Z_test"] - mean_src) / std_src
    test_tgt_z = (tgt_ae["Z_test"] - mean_tgt) / std_tgt

    train_src = train_src_z.astype(np.float32)
    train_tgt = train_tgt_z.astype(np.float32)
    input_dim = train_src.shape[1]
    output_dim = train_tgt.shape[1]

    if input_dim != output_dim:
        raise ValueError(
            f"Source and target latent dimensions must match for this CycleGAN: "
            f"{input_dim} vs {output_dim}"
        )

    latent_dim = input_dim
    print(f"Latent dimension inferred from AE outputs: {latent_dim}")

    tgt_bank = tf.constant(train_tgt, dtype=tf.float32)
    steps_per_epoch = max(len(train_tgt) // batch_size, 1)

    ds_src = tf.data.Dataset.from_tensor_slices(train_src).shuffle(8192).repeat()
    ds_tgt = tf.data.Dataset.from_tensor_slices(train_tgt).shuffle(8192).repeat()
    train_dataset = (tf.data.Dataset.zip((ds_src, ds_tgt))
                      .batch(batch_size, drop_remainder=True)
                      .prefetch(tf.data.AUTOTUNE))

    print(f"{SRC_NAME} train: {train_src.shape} | {TGT_NAME} train: {train_tgt.shape} "
          f"| steps/epoch: {steps_per_epoch}")

    # ---- Build models ---------------------------------------------------------
    G_SA_core = build_generator(input_dim, output_dim, "G_SA_core")
    inp = layers.Input(shape=(input_dim,))
    y = G_SA_core(inp)
    y = layers.Dense(output_dim, use_bias=True, name="linear_head")(y)
    G_SA = models.Model(inp, y, name="G_SA")

    G_AS = build_generator(output_dim, input_dim, "G_AS")
    D_Tgt = build_discriminator(output_dim, "D_Astro")
    D_Src = build_discriminator(input_dim, "D_iPSC")

    lr_sched = tf.keras.optimizers.schedules.ExponentialDecay(1e-4, 8000, 0.9, staircase=True)
    opt_SA = Adam(lr_sched, beta_1=0.5, clipnorm=1.0)
    opt_AS = Adam(lr_sched, beta_1=0.5, clipnorm=1.0)
    opt_DT = Adam(lr_sched, beta_1=0.5, clipnorm=1.0)
    opt_DS = Adam(lr_sched, beta_1=0.5, clipnorm=1.0)

    @tf.function
    def train_step(real_src, real_tgt):
        for _ in tf.range(N_CRITIC):
            with tf.GradientTape(persistent=True) as tD:
                fake_tgt = tf.stop_gradient(G_SA(real_src, training=True))
                fake_src = tf.stop_gradient(G_AS(real_tgt, training=True))
                rT = D_Tgt(real_tgt, training=True); fT = D_Tgt(fake_tgt, training=True)
                rS = D_Src(real_src, training=True); fS = D_Src(fake_src, training=True)
                gpT = gradient_penalty(D_Tgt, real_tgt, fake_tgt)
                gpS = gradient_penalty(D_Src, real_src, fake_src)
                dT = (discriminator_loss(rT, fT) + GP_WEIGHT * gpT
                      + DRIFT_EPS * (tf.reduce_mean(rT ** 2) + tf.reduce_mean(fT ** 2)))
                dS = (discriminator_loss(rS, fS) + GP_WEIGHT * gpS
                      + DRIFT_EPS * (tf.reduce_mean(rS ** 2) + tf.reduce_mean(fS ** 2)))
            opt_DT.apply_gradients(zip(tD.gradient(dT, D_Tgt.trainable_variables), D_Tgt.trainable_variables))
            opt_DS.apply_gradients(zip(tD.gradient(dS, D_Src.trainable_variables), D_Src.trainable_variables))
            del tD
        w_gap = tf.reduce_mean(rT) - tf.reduce_mean(fT)

        with tf.GradientTape(persistent=True) as tG:
            fake_tgt = G_SA(real_src, training=True)
            fake_src = G_AS(real_tgt, training=True)
            cyc_src = cycle_loss(real_src, G_AS(fake_tgt, training=True))
            cyc_tgt = cycle_loss(real_tgt, G_SA(fake_src, training=True))
            id_SA = identity_loss(real_tgt, G_SA(real_tgt, training=True))
            id_AS = identity_loss(real_src, G_AS(real_src, training=True))
            pair = knn_pair_loss_cos(fake_tgt, tgt_bank)
            mmd = mmd_rbf(fake_tgt, real_tgt)
            cval = cov_loss(fake_tgt, real_tgt)
            mom = moment_loss(fake_tgt, real_tgt)
            mom_as = moment_loss(fake_src, real_src)

            loss_SA = (generator_loss(D_Tgt(fake_tgt, training=True))
                       + LAMBDA_CYCLE * (cyc_src + cyc_tgt) + LAMBDA_IDENTITY * id_SA
                       + PAIR_W * pair + MMD_W * mmd + COV_W * cval + MOM_W * mom)
            loss_AS = (generator_loss(D_Src(fake_src, training=True))
                       + LAMBDA_CYCLE * (cyc_src + cyc_tgt) + LAMBDA_IDENTITY * id_AS
                       + MOM_W * mom_as)

        opt_SA.apply_gradients(zip(tG.gradient(loss_SA, G_SA.trainable_variables), G_SA.trainable_variables))
        opt_AS.apply_gradients(zip(tG.gradient(loss_AS, G_AS.trainable_variables), G_AS.trainable_variables))
        del tG

        return {"gen_SA": loss_SA, "gen_AS": loss_AS, "disc_Tgt": dT, "disc_Src": dS,
                "cycle": cyc_src + cyc_tgt, "identity": id_SA + id_AS,
                "pair": pair, "mmd": mmd, "cov": cval, "mom": mom, "mom_as": mom_as,
                "w_gap": w_gap}

    hist_keys = ["gen_SA", "gen_AS", "disc_Tgt", "disc_Src", "cycle", "identity",
                 "pair", "mmd", "cov", "mom", "mom_as", "w_gap"]
    histories = {k: [] for k in hist_keys}

    for ep in range(1, epochs + 1):
        ep_stats = {k: 0.0 for k in hist_keys}
        steps = 0
        for r_src, r_tgt in train_dataset.take(steps_per_epoch):
            out = train_step(r_src, r_tgt)
            for k in ep_stats:
                ep_stats[k] += float(out[k])
            steps += 1
        for k in histories:
            histories[k].append(ep_stats[k] / max(steps, 1))
        if ep % 10 == 0 or ep == 1:
            print(f"[{drug_display_name}] Epoch {ep:03d} | W-gap={histories['w_gap'][-1]:.3f} | "
                  f"genSA={histories['gen_SA'][-1]:.3f} | mmd={histories['mmd'][-1]:.3f} | "
                  f"cycle={histories['cycle'][-1]:.3f} | mom={histories['mom'][-1]:.3f}")

    G_SA.save_weights(os.path.join(drug_out_dir, "G_SA.weights.h5"))
    G_AS.save_weights(os.path.join(drug_out_dir, "G_AS.weights.h5"))
    pd.DataFrame(histories).to_csv(
        os.path.join(drug_out_dir, "training_history.csv"), index=False
    )

    np.savez(
        os.path.join(drug_out_dir, "latent_scaler_stats.npz"),
        mean_src=mean_src,
        std_src=std_src,
        mean_tgt=mean_tgt,
        std_tgt=std_tgt,
    )

    print(f"Weights + history + latent scaler stats saved -> {drug_out_dir}")

    # ---- Translate test set (iPSC -> Astro) -----------------------------------
    fake_tgt_test_z = G_SA(
        tf.constant(test_src_z.astype(np.float32)), training=False
    ).numpy()
    fake_tgt_test_lat = fake_tgt_test_z * std_tgt + mean_tgt

    np.save(os.path.join(drug_out_dir, "fake_tgt_test_std.npy"), fake_tgt_test_z)
    np.save(os.path.join(drug_out_dir, "fake_tgt_test_latent.npy"), fake_tgt_test_lat)
    tgt_ckpt=torch.load(os.path.join(ae_dir,TGT_NAME+'_ae_checkpoint.pt'),map_location='cpu',weights_only=False)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    target_model=MorphologyAutoencoder(len(tgt_ae['feature_cols']),int(tgt_ckpt['config']['latent_dim'])).to(device)
    target_model.load_state_dict(tgt_ckpt['model_state_dict']);target_model.eval()
    decoded=[]
    with torch.no_grad():
        for batch in np.array_split(fake_tgt_test_lat.astype(np.float32),max(1,int(np.ceil(len(fake_tgt_test_lat)/256)))):
            decoded.append(target_model.decoder(torch.as_tensor(batch,device=device)).cpu().numpy())
    fake_scaled=np.concatenate(decoded,axis=0)
    feature_stats=tgt_ae['scaler_stats']
    fake_original=fake_scaled*feature_stats['scaler_scale'].to_numpy()+feature_stats['scaler_mean'].to_numpy()
    assert fake_original.shape[1]==len(tgt_ae['feature_cols']) and np.isfinite(fake_original).all()
    np.save(os.path.join(drug_out_dir,'translated_astro_morphology_original.npy'),fake_original.astype(np.float32))
    pd.DataFrame({'feature':tgt_ae['feature_cols']}).to_csv(os.path.join(drug_out_dir,'translated_astro_feature_names.csv'),index=False)
    np.save(os.path.join(drug_out_dir,'source_test_idx.npy'),src_ae['te_idx'])
    real_tgt_lat = tgt_ae["Z_test"].astype(np.float32)
    fake_tgt_lat = fake_tgt_test_lat.astype(np.float32)

    def safe_corr(a, b):
        a = np.asarray(a, dtype=np.float64)
        b = np.asarray(b, dtype=np.float64)
        if np.std(a) < 1e-12 or np.std(b) < 1e-12:
            return np.nan
        return float(pearsonr(a, b)[0])

    latent_mean_r = safe_corr(real_tgt_lat.mean(0), fake_tgt_lat.mean(0))
    latent_sd_r = safe_corr(real_tgt_lat.std(0), fake_tgt_lat.std(0))

    # KS is scale-invariant, so use the standardized target coordinates.
    ks_vals = np.array([
        ks_2samp(test_tgt_z[:, g], fake_tgt_test_z[:, g]).statistic
        for g in range(test_tgt_z.shape[1])
    ], dtype=np.float32)

    # W1 is scale-dependent; use standardized target coordinates so dimensions
    # are comparable.
    w1_vals = np.array([
        wasserstein_distance(test_tgt_z[:, g], fake_tgt_test_z[:, g])
        for g in range(test_tgt_z.shape[1])
    ], dtype=np.float32)

    mean_abs_diff_std = float(np.mean(np.abs(
        test_tgt_z.mean(0) - fake_tgt_test_z.mean(0)
    )))
    sd_abs_diff_std = float(np.mean(np.abs(
        test_tgt_z.std(0) - fake_tgt_test_z.std(0)
    )))

    print(f"\n=== {drug_display_name}: iPSC -> Astro validation (latent space) ===")
    print(f"Latent mean correlation: {latent_mean_r:.4f}")
    print(f"Latent SD correlation:   {latent_sd_r:.4f}")
    print(f"Mean |mean diff| (std):  {mean_abs_diff_std:.4f}")
    print(f"Mean |SD diff| (std):    {sd_abs_diff_std:.4f}")
    print(f"Median KS:               {np.median(ks_vals):.4f}")
    print(f"Median W1 (std latent):  {np.median(w1_vals):.4f}")

    pd.DataFrame({
        "latent_dim": np.arange(test_tgt_z.shape[1]),
        "KS": ks_vals,
        "W1_std": w1_vals,
    }).to_csv(
        os.path.join(drug_out_dir, "latent_distribution_metrics.csv"),
        index=False,
    )

    if save_plots:
        rng = np.random.default_rng(42)
        n_real = min(2000, len(real_tgt_lat))
        n_fake = min(2000, len(fake_tgt_lat))
        real_idx = rng.choice(len(real_tgt_lat), n_real, replace=False)
        fake_idx = rng.choice(len(fake_tgt_lat), n_fake, replace=False)

        Z_stack = np.vstack([real_tgt_lat[real_idx], fake_tgt_lat[fake_idx]])
        lb = np.array([f"{TGT_NAME} (Target)"] * n_real + ["Translated"] * n_fake)

        emb = umap.UMAP(
            n_components=2,
            n_neighbors=30,
            min_dist=0.3,
            random_state=42,
        ).fit_transform(Z_stack)

        fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
        colors = {f"{TGT_NAME} (Target)": "#9467BD", "Translated": "#DD8452"}

        for grp in [f"{TGT_NAME} (Target)", "Translated"]:
            m = lb == grp
            mk = "X" if grp == "Translated" else "o"
            ax.scatter(
                emb[m, 0], emb[m, 1],
                c=colors[grp],
                marker=mk,
                s=14 if mk == "X" else 8,
                alpha=0.8 if mk == "X" else 0.5,
                linewidths=0,
                label=f"{grp} (n={m.sum():,})",
            )

        ax.set_title(f"{drug_display_name}: real vs translated {TGT_NAME} latent")
        ax.legend(fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        plt.tight_layout()
        plt.savefig(
            os.path.join(drug_out_dir, f"Fig_umap_target_vs_translated_{slug}.png"),
            dpi=300,
            bbox_inches="tight",
        )
        plt.close(fig)

    return {
        "drug": drug_display_name,
        "slug": slug,
        "latent_dim": latent_dim,
        "n_src_test": len(src_ae["Z_test"]),
        "n_tgt_test": len(tgt_ae["Z_test"]),
        "latent_mean_r": latent_mean_r,
        "latent_sd_r": latent_sd_r,
        "mean_abs_diff_std": mean_abs_diff_std,
        "sd_abs_diff_std": sd_abs_diff_std,
        "median_KS": float(np.median(ks_vals)),
        "median_W1_std": float(np.median(w1_vals)),
        "out_dir": drug_out_dir,
    }




### Developmental output saving (implemented inside `run_drug_translation`)

For each of DMSO, Latrunculin_B, GW3965 and Clozapine, the function automatically: translates **test iPSC latents** with `G_SA`, reverses target latent standardization, loads the **corresponding frozen astrocyte AE** checkpoint, decodes into target-scaled morphological features, reverses target morphology scaling, then saves `translated_astro_morphology_original.npy`, `translated_astro_feature_names.csv`, and `source_test_idx.npy` under that condition’s directory. The files are written before latent-space test metrics. No additional decoding cell is needed. **Do not use test-set results to select model checkpoints.**

In [ ]:
dev_results=[]
for drug in ALL_CONDITIONS:
    print('DEVELOPMENTAL TRANSLATION',drug)
    dev_results.append(run_drug_translation(drug,ae_dir=AE_DIR,out_root=DEV_DIR,save_plots=True))
pd.DataFrame(dev_results).to_csv(os.path.join(DEV_DIR,'ipsc_astro_drug_translation_summary.csv'),index=False)


## 5. Train six DMSO → drug translators (iPSC and astrocytes)

In [ ]:
# =============================================================================
# DMSO -> DRUG MORPHOLOGY TRANSLATION
#
# Six CycleGAN models:
#
#   iPSC_DMSO       <-> iPSC_Latrunculin
#   iPSC_DMSO       <-> iPSC_GW3965
#   iPSC_DMSO       <-> iPSC_Clozapine
#
#   Astrocytes_DMSO <-> Astrocytes_Latrunculin
#   Astrocytes_DMSO <-> Astrocytes_GW3965
#   Astrocytes_DMSO <-> Astrocytes_Clozapine
#
# Forward direction is the biological prediction of interest:
#
#       DMSO -> drug-treated state
#
# Reverse generator is trained for CycleGAN consistency.
# =============================================================================

import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input, Dense, LayerNormalization
from tensorflow.keras.optimizers import Adam

import matplotlib.pyplot as plt

BASE_DIR=str(RUN_ROOT)
AE_DIR=str(RUN_ROOT / 'autoencoders')
OUT_ROOT=CTRL_DIR
os.makedirs(OUT_ROOT,exist_ok=True)

DRUGS = [
    "Latrunculin_B",
    "GW3965",
    "Clozapine",
]


DRUG_SLUGS = {
    "Latrunculin_B": "Latrunculin_B",
    "GW3965": "GW3965",
    "Clozapine": "Clozapine",
}


CELL_TYPES = [
    "iPSC",
    "Astrocytes",
]


SEED = 42

BATCH_SIZE = 32
EPOCHS = 300

GP_WEIGHT = 10.0
DRIFT_EPS = 1e-3
N_CRITIC = 5

LAMBDA_CYCLE = 5.0
LAMBDA_IDENTITY = 0.5

PAIR_W = 1.0
MMD_W = 1.0
COV_W = 0.2
MOM_W = 2.0

K_PAIR = 5
BANK_SAMPLE = 8192

LR_INITIAL = 1e-4
LR_DECAY_STEPS = 8000
LR_DECAY_RATE = 0.9
ADAM_BETA1 = 0.5
CLIPNORM = 1.0

In [ ]:

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    tf.random.set_seed(seed)


set_seed(SEED)



try:

    tf_gpus = tf.config.list_physical_devices("GPU")

    for gpu in tf_gpus:
        tf.config.experimental.set_memory_growth(
            gpu,
            True
        )

except RuntimeError:
    pass

In [ ]:

# MORPHOLOGY AUTOENCODER

class MorphologyAutoencoder(nn.Module):

    def __init__(
        self,
        input_dim,
        latent_dim=128
    ):

        super().__init__()

        def enc_block(d_in, d_out):

            return nn.Sequential(
                nn.Linear(d_in, d_out),
                nn.BatchNorm1d(d_out),
                nn.LeakyReLU(0.1),
                nn.Dropout(0.15),
            )

        def dec_block(
            d_in,
            d_out,
            drop=True
        ):

            layers_ = [
                nn.Linear(d_in, d_out),
                nn.BatchNorm1d(d_out),
                nn.LeakyReLU(0.1),
            ]

            if drop:
                layers_.append(
                    nn.Dropout(0.15)
                )

            return nn.Sequential(
                *layers_
            )


        self.encoder = nn.Sequential(

            enc_block(
                input_dim,
                1024
            ),

            enc_block(
                1024,
                512
            ),

            nn.Sequential(
                nn.Linear(512, 256),
                nn.BatchNorm1d(256),
                nn.LeakyReLU(0.1),
            ),

            nn.Linear(
                256,
                latent_dim
            ),
        )


        self.decoder = nn.Sequential(

            dec_block(
                latent_dim,
                256
            ),

            dec_block(
                256,
                512
            ),

            dec_block(
                512,
                1024,
                drop=False
            ),

            nn.Linear(
                1024,
                input_dim
            ),
        )


    def forward(self, x):

        z = self.encoder(x)
        recon = self.decoder(z)

        return z, recon


    def encode(self, x):

        return self.encoder(x)

In [ ]:

def load_condition_ae(
    name,
    ae_dir,
    device
):

    prefix = os.path.join(
        ae_dir,
        name
    )


    required = [
        "_scaler_stats.csv",
        "_Z_train.npy",
        "_Z_test.npy",
        "_X_train_scaled.npy",
        "_X_test_scaled.npy",
        "_train_idx.npy",
        "_test_idx.npy",
        "_ae_checkpoint.pt",
    ]


    missing = [
        suffix
        for suffix in required
        if not os.path.exists(
            prefix + suffix
        )
    ]


    if missing:

        raise FileNotFoundError(
            f"{name}: missing files:\n"
            + "\n".join(missing)
        )
    # Feature/scaler information

    scaler_stats = pd.read_csv(
        prefix + "_scaler_stats.csv"
    )

    feature_cols = (
        scaler_stats["feature"]
        .tolist()
    )
    # AE checkpoint

    ckpt = torch.load(
        prefix + "_ae_checkpoint.pt",
        map_location="cpu",
        weights_only=False,
    )


    input_dim = len(
        feature_cols
    )

    latent_dim = int(
        ckpt["config"]["latent_dim"]
    )


    model = MorphologyAutoencoder(
        input_dim=input_dim,
        latent_dim=latent_dim,
    ).to(device)


    model.load_state_dict(
        ckpt["model_state_dict"]
    )

    model.eval()

    Z_train = np.load(
        prefix + "_Z_train.npy"
    ).astype(np.float32)

    Z_test = np.load(
        prefix + "_Z_test.npy"
    ).astype(np.float32)

    X_train = np.load(
        prefix + "_X_train_scaled.npy"
    ).astype(np.float32)

    X_test = np.load(
        prefix + "_X_test_scaled.npy"
    ).astype(np.float32)

    train_idx = np.load(
        prefix + "_train_idx.npy"
    )

    test_idx = np.load(
        prefix + "_test_idx.npy"
    )

    mean_lat = Z_train.mean(
        axis=0,
        keepdims=True
    )

    std_lat = Z_train.std(
        axis=0,
        keepdims=True
    )

    std_lat = np.maximum(
        std_lat,
        1e-6
    )


    return {

        "name":
            name,

        "model":
            model,

        "input_dim":
            input_dim,

        "latent_dim":
            latent_dim,

        "feature_cols":
            feature_cols,

        "Z_train":
            Z_train,

        "Z_test":
            Z_test,

        "X_train":
            X_train,

        "X_test":
            X_test,

        "train_idx":
            train_idx,

        "test_idx":
            test_idx,

        "mean_lat":
            mean_lat,

        "std_lat":
            std_lat,

        "scaler_mean":
            scaler_stats[
                "scaler_mean"
            ].values.astype(
                np.float32
            ),

        "scaler_scale":
            scaler_stats[
                "scaler_scale"
            ].values.astype(
                np.float32
            ),

        "train_median":
            (
                scaler_stats[
                    "train_median"
                ].values.astype(
                    np.float32
                )

                if "train_median"
                in scaler_stats.columns

                else None
            ),
    }

In [ ]:

def standardize_latents(
    Z,
    mean_lat,
    std_lat
):

    return (
        (Z - mean_lat)
        / std_lat
    ).astype(
        np.float32
    )

In [ ]:

def build_generator(
    d_in,
    d_out,
    name
):

    x = inp = Input(
        shape=(d_in,)
    )

    x = Dense(
        512,
        activation="relu"
    )(x)

    x = LayerNormalization()(x)


    x = Dense(
        256,
        activation="relu"
    )(x)

    x = LayerNormalization()(x)


    x = Dense(
        256,
        activation="relu"
    )(x)

    x = LayerNormalization()(x)


    x = Dense(
        d_out
    )(x)


    return models.Model(
        inp,
        x,
        name=name
    )


def build_forward_generator(
    d_in,
    d_out
):

    core = build_generator(
        d_in,
        d_out,
        "G_control_to_drug_core"
    )


    inp = layers.Input(
        shape=(d_in,)
    )

    y = core(
        inp
    )


    # forward affine calibration head
    y = layers.Dense(
        d_out,
        use_bias=True,
        name="linear_head"
    )(y)


    return models.Model(
        inp,
        y,
        name="G_control_to_drug"
    )


def build_discriminator(
    d,
    name
):

    x = inp = Input(
        shape=(d,)
    )

    x = Dense(
        256,
        activation="relu"
    )(x)

    x = Dense(
        128,
        activation="relu"
    )(x)

    x = Dense(
        64,
        activation="relu"
    )(x)

    x = Dense(
        1
    )(x)


    return models.Model(
        inp,
        x,
        name=name
    )

In [ ]:

def generator_loss(
    fake_scores
):

    return -tf.reduce_mean(
        fake_scores
    )


def discriminator_loss(
    real_scores,
    fake_scores
):

    return (
        tf.reduce_mean(
            fake_scores
        )
        -
        tf.reduce_mean(
            real_scores
        )
    )


def cycle_loss(
    real,
    cycled
):

    return tf.reduce_mean(
        tf.abs(
            real - cycled
        )
    )


def identity_loss(
    real,
    same
):

    return tf.reduce_mean(
        tf.abs(
            real - same
        )
    )


def gradient_penalty(
    discriminator,
    real,
    fake
):

    alpha = tf.random.uniform(
        [
            tf.shape(real)[0],
            1
        ]
    )


    interpolated = (
        real
        +
        alpha
        *
        (
            fake - real
        )
    )


    with tf.GradientTape() as tape:

        tape.watch(
            interpolated
        )

        prediction = discriminator(
            interpolated,
            training=True
        )


    gradient = tape.gradient(
        prediction,
        interpolated
    )


    norm = tf.sqrt(
        tf.reduce_sum(
            tf.square(
                gradient
            ),
            axis=1
        )
        +
        1e-12
    )


    return tf.reduce_mean(
        tf.square(
            norm - 1.0
        )
    )

In [ ]:
def knn_pair_loss_cos(
    fake,
    bank,
    k=5,
    bank_sample=8192
):

    n = tf.shape(
        bank
    )[0]


    m = tf.minimum(
        bank_sample,
        n
    )


    idx = tf.random.uniform(
        shape=[m],
        minval=0,
        maxval=n,
        dtype=tf.int32
    )


    B = tf.gather(
        bank,
        idx
    )


    f = tf.math.l2_normalize(
        fake,
        axis=1
    )

    b = tf.math.l2_normalize(
        B,
        axis=1
    )


    similarity = tf.matmul(
        f,
        b,
        transpose_b=True
    )


    k_use = tf.minimum(
        tf.cast(k, tf.int32),
        m
    )


    _, indices = tf.math.top_k(
        similarity,
        k=k_use
    )


    b_knn = tf.gather(
        b,
        indices,
        axis=0
    )


    target_neighbor = tf.reduce_mean(
        b_knn,
        axis=1
    )


    return tf.reduce_mean(
        1.0
        -
        tf.reduce_sum(
            f * target_neighbor,
            axis=1
        )
    )

In [ ]:
def mmd_rbf(
    x,
    y,
    sigmas=(0.5, 1.0, 2.0)
):

    def kernel(
        a,
        b,
        sigma
    ):

        a2 = tf.reduce_sum(
            a * a,
            axis=1,
            keepdims=True
        )

        b2 = tf.reduce_sum(
            b * b,
            axis=1,
            keepdims=True
        )


        d2 = (
            a2
            -
            2.0
            *
            tf.matmul(
                a,
                b,
                transpose_b=True
            )
            +
            tf.transpose(
                b2
            )
        )


        return tf.exp(
            -d2
            /
            (
                2.0
                *
                sigma
                *
                sigma
            )
        )


    kxx = tf.add_n([
        kernel(
            x,
            x,
            s
        )
        for s in sigmas
    ]) / len(sigmas)


    kyy = tf.add_n([
        kernel(
            y,
            y,
            s
        )
        for s in sigmas
    ]) / len(sigmas)


    kxy = tf.add_n([
        kernel(
            x,
            y,
            s
        )
        for s in sigmas
    ]) / len(sigmas)


    return (
        tf.reduce_mean(
            kxx
        )
        +
        tf.reduce_mean(
            kyy
        )
        -
        2.0
        *
        tf.reduce_mean(
            kxy
        )
    )

In [ ]:
def covariance_matrix(
    x
):

    xc = (
        x
        -
        tf.reduce_mean(
            x,
            axis=0,
            keepdims=True
        )
    )


    denom = tf.cast(
        tf.maximum(
            tf.shape(xc)[0] - 1,
            1
        ),
        tf.float32
    )


    return (
        tf.matmul(
            xc,
            xc,
            transpose_a=True
        )
        /
        denom
    )


def cov_loss(
    fake,
    real
):

    return tf.reduce_mean(
        tf.square(
            covariance_matrix(fake)
            -
            covariance_matrix(real)
        )
    )


def moment_loss(
    fake,
    real
):

    mean_loss = tf.reduce_mean(
        tf.abs(
            tf.reduce_mean(
                fake,
                axis=0
            )
            -
            tf.reduce_mean(
                real,
                axis=0
            )
        )
    )


    std_loss = tf.reduce_mean(
        tf.abs(
            tf.math.reduce_std(
                fake,
                axis=0
            )
            -
            tf.math.reduce_std(
                real,
                axis=0
            )
        )
    )


    return (
        mean_loss
        +
        std_loss
    )

In [ ]:

def decode_latents(
    model,
    Z_np,
    device,
    batch_size=256
):

    ds = TensorDataset(
        torch.tensor(
            Z_np,
            dtype=torch.float32
        )
    )


    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False
    )


    out = []

    model.eval()


    with torch.no_grad():

        for (z,) in dl:

            recon = model.decoder(
                z.to(
                    device
                )
            )

            out.append(
                recon
                .cpu()
                .numpy()
            )


    return np.concatenate(
        out,
        axis=0
    )

In [ ]:

def run_control_to_drug_translation(
    cell_type,
    drug_display_name,
    ae_dir=AE_DIR,
    out_root=OUT_ROOT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
):

    # clear previous TensorFlow graph/models
    tf.keras.backend.clear_session()

    set_seed(
        SEED
    )


    slug = DRUG_SLUGS[
        drug_display_name
    ]

    SRC_NAME = (
        f"{cell_type}_DMSO"
    )

    TGT_NAME = (
        f"{cell_type}_{slug}"
    )


    model_name = (
        f"{cell_type}_DMSO_to_{slug}"
    )


    model_out = os.path.join(
        out_root,
        model_name
    )


    os.makedirs(
        model_out,
        exist_ok=True
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        model_name
    )

    print(
        f"{SRC_NAME} -> {TGT_NAME}"
    )

    print(
        "=" * 80
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    src_ae = load_condition_ae(
        SRC_NAME,
        ae_dir,
        device
    )

    tgt_ae = load_condition_ae(
        TGT_NAME,
        ae_dir,
        device
    )


    print(
        f"{SRC_NAME}: "
        f"{len(src_ae['feature_cols'])} morphology features | "
        f"latent={src_ae['latent_dim']}"
    )


    print(
        f"{TGT_NAME}: "
        f"{len(tgt_ae['feature_cols'])} morphology features | "
        f"latent={tgt_ae['latent_dim']}"
    )

    n_common = len(
        set(src_ae["feature_cols"])
        &
        set(tgt_ae["feature_cols"])
    )


    print(
        f"Common named morphology features: {n_common}"
    )


    # Cycle consistency/identity requires matching latent dimensionality
    assert (
        src_ae["latent_dim"]
        ==
        tgt_ae["latent_dim"]
    ), (
        f"Latent dimension mismatch: "
        f"{src_ae['latent_dim']} vs "
        f"{tgt_ae['latent_dim']}"
    )


    latent_dim = (
        src_ae["latent_dim"]
    )

    # Standardize latent spaces using each condition's TRAIN latents only


    train_src = standardize_latents(
        src_ae["Z_train"],
        src_ae["mean_lat"],
        src_ae["std_lat"]
    )


    train_tgt = standardize_latents(
        tgt_ae["Z_train"],
        tgt_ae["mean_lat"],
        tgt_ae["std_lat"]
    )


    test_src = standardize_latents(
        src_ae["Z_test"],
        src_ae["mean_lat"],
        src_ae["std_lat"]
    )


    test_tgt = standardize_latents(
        tgt_ae["Z_test"],
        tgt_ae["mean_lat"],
        tgt_ae["std_lat"]
    )


    print(
        "Source train latent:",
        train_src.shape
    )

    print(
        "Target train latent:",
        train_tgt.shape
    )

    print(
        "Source test latent:",
        test_src.shape
    )

    print(
        "Target test latent:",
        test_tgt.shape
    )
    # Target bank for forward kNN pair loss

    tgt_bank = tf.constant(
        train_tgt,
        dtype=tf.float32
    )
    steps_per_epoch = max(
        len(train_tgt)
        // batch_size,
        1
    )
    ds_src = (
        tf.data.Dataset
        .from_tensor_slices(
            train_src
        )
        .shuffle(
            min(
                len(train_src),
                8192
            ),
            seed=SEED,
            reshuffle_each_iteration=True,
        )
        .repeat()
    )


    ds_tgt = (
        tf.data.Dataset
        .from_tensor_slices(
            train_tgt
        )
        .shuffle(
            min(
                len(train_tgt),
                8192
            ),
            seed=SEED + 1,
            reshuffle_each_iteration=True,
        )
        .repeat()
    )


    train_dataset = (
        tf.data.Dataset
        .zip(
            (
                ds_src,
                ds_tgt
            )
        )
        .batch(
            batch_size,
            drop_remainder=True
        )
        .prefetch(
            tf.data.AUTOTUNE
        )
    )


    print(
        "steps_per_epoch:",
        steps_per_epoch
    )

    G_CT = build_forward_generator(
        latent_dim,
        latent_dim
    )


    G_TC = build_generator(
        latent_dim,
        latent_dim,
        "G_drug_to_control"
    )


    D_TGT = build_discriminator(
        latent_dim,
        "D_drug"
    )


    D_SRC = build_discriminator(
        latent_dim,
        "D_control"
    )

    lr_schedule = (
        tf.keras.optimizers
        .schedules
        .ExponentialDecay(
            initial_learning_rate=LR_INITIAL,
            decay_steps=LR_DECAY_STEPS,
            decay_rate=LR_DECAY_RATE,
            staircase=True,
        )
    )


    opt_CT = Adam(
        learning_rate=lr_schedule,
        beta_1=ADAM_BETA1,
        clipnorm=CLIPNORM,
    )


    opt_TC = Adam(
        learning_rate=lr_schedule,
        beta_1=ADAM_BETA1,
        clipnorm=CLIPNORM,
    )


    opt_DT = Adam(
        learning_rate=lr_schedule,
        beta_1=ADAM_BETA1,
        clipnorm=CLIPNORM,
    )


    opt_DS = Adam(
        learning_rate=lr_schedule,
        beta_1=ADAM_BETA1,
        clipnorm=CLIPNORM,
    )

    @tf.function
    def train_step(
        real_control,
        real_drug
    ):

        for _ in tf.range(
            N_CRITIC
        ):

            with tf.GradientTape(
                persistent=True
            ) as tape_D:


                # Generator outputs detached from critic update
                fake_drug = tf.stop_gradient(
                    G_CT(
                        real_control,
                        training=True
                    )
                )


                fake_control = tf.stop_gradient(
                    G_TC(
                        real_drug,
                        training=True
                    )
                )


                # target/drug critic
                rT = D_TGT(
                    real_drug,
                    training=True
                )

                fT = D_TGT(
                    fake_drug,
                    training=True
                )


                # source/control critic
                rS = D_SRC(
                    real_control,
                    training=True
                )

                fS = D_SRC(
                    fake_control,
                    training=True
                )


                gpT = gradient_penalty(
                    D_TGT,
                    real_drug,
                    fake_drug
                )


                gpS = gradient_penalty(
                    D_SRC,
                    real_control,
                    fake_control
                )


                dT = (
                    discriminator_loss(
                        rT,
                        fT
                    )
                    +
                    GP_WEIGHT
                    *
                    gpT
                    +
                    DRIFT_EPS
                    *
                    (
                        tf.reduce_mean(
                            rT ** 2
                        )
                        +
                        tf.reduce_mean(
                            fT ** 2
                        )
                    )
                )


                dS = (
                    discriminator_loss(
                        rS,
                        fS
                    )
                    +
                    GP_WEIGHT
                    *
                    gpS
                    +
                    DRIFT_EPS
                    *
                    (
                        tf.reduce_mean(
                            rS ** 2
                        )
                        +
                        tf.reduce_mean(
                            fS ** 2
                        )
                    )
                )


            grads_DT = tape_D.gradient(
                dT,
                D_TGT.trainable_variables
            )


            grads_DS = tape_D.gradient(
                dS,
                D_SRC.trainable_variables
            )


            opt_DT.apply_gradients(
                zip(
                    grads_DT,
                    D_TGT.trainable_variables
                )
            )


            opt_DS.apply_gradients(
                zip(
                    grads_DS,
                    D_SRC.trainable_variables
                )
            )


            del tape_D


        wasserstein_gap = (
            tf.reduce_mean(rT)
            -
            tf.reduce_mean(fT)
        )

        with tf.GradientTape(
            persistent=True
        ) as tape_G:

            fake_drug = G_CT(
                real_control,
                training=True
            )


            fake_control = G_TC(
                real_drug,
                training=True
            )

            adv_CT = generator_loss(
                D_TGT(
                    fake_drug,
                    training=True
                )
            )


            adv_TC = generator_loss(
                D_SRC(
                    fake_control,
                    training=True
                )
            )

            cycled_control = G_TC(
                fake_drug,
                training=True
            )


            cycled_drug = G_CT(
                fake_control,
                training=True
            )


            cyc_control = cycle_loss(
                real_control,
                cycled_control
            )


            cyc_drug = cycle_loss(
                real_drug,
                cycled_drug
            )


            cycle_total = (
                cyc_control
                +
                cyc_drug
            )

            same_drug = G_CT(
                real_drug,
                training=True
            )


            same_control = G_TC(
                real_control,
                training=True
            )


            id_CT = identity_loss(
                real_drug,
                same_drug
            )


            id_TC = identity_loss(
                real_control,
                same_control
            )

            pair = knn_pair_loss_cos(
                fake_drug,
                tgt_bank,
                k=K_PAIR,
                bank_sample=BANK_SAMPLE
            )


            mmd = mmd_rbf(
                fake_drug,
                real_drug
            )


            cval = cov_loss(
                fake_drug,
                real_drug
            )


            mom = moment_loss(
                fake_drug,
                real_drug
            )

            mom_reverse = moment_loss(
                fake_control,
                real_control
            )

            loss_CT = (
                adv_CT

                +
                LAMBDA_CYCLE
                *
                cycle_total

                +
                LAMBDA_IDENTITY
                *
                id_CT

                +
                PAIR_W
                *
                pair

                +
                MMD_W
                *
                mmd

                +
                COV_W
                *
                cval

                +
                MOM_W
                *
                mom
            )


            loss_TC = (
                adv_TC

                +
                LAMBDA_CYCLE
                *
                cycle_total

                +
                LAMBDA_IDENTITY
                *
                id_TC

                +
                MOM_W
                *
                mom_reverse
            )


        grads_CT = tape_G.gradient(
            loss_CT,
            G_CT.trainable_variables
        )


        grads_TC = tape_G.gradient(
            loss_TC,
            G_TC.trainable_variables
        )


        opt_CT.apply_gradients(
            zip(
                grads_CT,
                G_CT.trainable_variables
            )
        )


        opt_TC.apply_gradients(
            zip(
                grads_TC,
                G_TC.trainable_variables
            )
        )


        del tape_G


        return {

            "gen_control_to_drug":
                loss_CT,

            "gen_drug_to_control":
                loss_TC,

            "disc_drug":
                dT,

            "disc_control":
                dS,

            "cycle":
                cycle_total,

            "identity":
                id_CT + id_TC,

            "pair":
                pair,

            "mmd":
                mmd,

            "cov":
                cval,

            "moment":
                mom,

            "moment_reverse":
                mom_reverse,

            "wasserstein_gap":
                wasserstein_gap,
        }

    history_keys = [

        "gen_control_to_drug",
        "gen_drug_to_control",

        "disc_drug",
        "disc_control",

        "cycle",
        "identity",

        "pair",
        "mmd",
        "cov",

        "moment",
        "moment_reverse",

        "wasserstein_gap",
    ]


    histories = {
        k: []
        for k in history_keys
    }


    for epoch in range(
        1,
        epochs + 1
    ):


        epoch_stats = {
            k: 0.0
            for k in history_keys
        }


        steps = 0


        for (
            real_control,
            real_drug
        ) in train_dataset.take(
            steps_per_epoch
        ):


            out = train_step(
                real_control,
                real_drug
            )


            for k in epoch_stats:

                epoch_stats[k] += float(
                    out[k]
                )


            steps += 1


        for k in histories:

            histories[k].append(
                epoch_stats[k]
                /
                max(
                    steps,
                    1
                )
            )


        if (
            epoch == 1
            or
            epoch % 10 == 0
        ):

            print(
                f"[{model_name}] "
                f"Epoch {epoch:03d}/{epochs} | "
                f"W-gap={histories['wasserstein_gap'][-1]:.3f} | "
                f"G={histories['gen_control_to_drug'][-1]:.3f} | "
                f"cycle={histories['cycle'][-1]:.3f} | "
                f"MMD={histories['mmd'][-1]:.3f} | "
                f"moment={histories['moment'][-1]:.3f}"
            )

    G_CT.save_weights(
        os.path.join(
            model_out,
            "G_control_to_drug.weights.h5"
        )
    )


    G_TC.save_weights(
        os.path.join(
            model_out,
            "G_drug_to_control.weights.h5"
        )
    )


    pd.DataFrame(
        histories
    ).to_csv(
        os.path.join(
            model_out,
            "training_history.csv"
        ),
        index=False
    )

    np.savez(
        os.path.join(
            model_out,
            "latent_standardization_stats.npz"
        ),

        src_mean=src_ae["mean_lat"],
        src_std=src_ae["std_lat"],

        tgt_mean=tgt_ae["mean_lat"],
        tgt_std=tgt_ae["std_lat"],
    )


    # FORWARD TRANSLATION
    # DMSO test cells -> predicted drug-treated cells

    fake_drug_std = G_CT(
        tf.constant(
            test_src,
            dtype=tf.float32
        ),
        training=False
    ).numpy()


    # standardized target latent -> target AE latent scale
    fake_drug_lat = (
        fake_drug_std
        *
        tgt_ae["std_lat"]
        +
        tgt_ae["mean_lat"]
    )


    # decode through drug-specific target AE
    X_fake_drug_scaled = decode_latents(
        tgt_ae["model"],
        fake_drug_lat,
        device
    )


    # restore original CellProfiler scale
    X_fake_drug_orig = (
        X_fake_drug_scaled
        *
        tgt_ae["scaler_scale"]
        +
        tgt_ae["scaler_mean"]
    )
    # REVERSE TRANSLATION
    # drug test -> reconstructed/predicted DMSO state

    fake_control_std = G_TC(
        tf.constant(
            test_tgt,
            dtype=tf.float32
        ),
        training=False
    ).numpy()


    fake_control_lat = (
        fake_control_std
        *
        src_ae["std_lat"]
        +
        src_ae["mean_lat"]
    )


    X_fake_control_scaled = decode_latents(
        src_ae["model"],
        fake_control_lat,
        device
    )


    X_fake_control_orig = (
        X_fake_control_scaled
        *
        src_ae["scaler_scale"]
        +
        src_ae["scaler_mean"]
    )


    np.save(
        os.path.join(
            model_out,
            "X_control_to_drug_scaled.npy"
        ),
        X_fake_drug_scaled
    )


    np.save(
        os.path.join(
            model_out,
            "X_control_to_drug_original.npy"
        ),
        X_fake_drug_orig
    )


    np.save(
        os.path.join(
            model_out,
            "Z_control_to_drug_std.npy"
        ),
        fake_drug_std
    )


    np.save(
        os.path.join(
            model_out,
            "Z_control_to_drug_latent.npy"
        ),
        fake_drug_lat
    )


    np.save(
        os.path.join(
            model_out,
            "X_drug_to_control_scaled.npy"
        ),
        X_fake_control_scaled
    )


    np.save(
        os.path.join(
            model_out,
            "X_drug_to_control_original.npy"
        ),
        X_fake_control_orig
    )


    # target feature names correspond to forward translated matrix
    pd.DataFrame({
        "feature":
            tgt_ae["feature_cols"]
    }).to_csv(
        os.path.join(
            model_out,
            "control_to_drug_feature_names.csv"
        ),
        index=False
    )


    # source feature names correspond to reverse translated matrix
    pd.DataFrame({
        "feature":
            src_ae["feature_cols"]
    }).to_csv(
        os.path.join(
            model_out,
            "drug_to_control_feature_names.csv"
        ),
        index=False
    )

    manifest = {

        "cell_type":
            cell_type,

        "drug":
            drug_display_name,

        "source":
            SRC_NAME,

        "target":
            TGT_NAME,

        "latent_dim":
            int(latent_dim),

        "source_n_features":
            int(
                len(
                    src_ae["feature_cols"]
                )
            ),

        "target_n_features":
            int(
                len(
                    tgt_ae["feature_cols"]
                )
            ),

        "common_features":
            int(n_common),

        "source_train_cells":
            int(
                len(
                    train_src
                )
            ),

        "target_train_cells":
            int(
                len(
                    train_tgt
                )
            ),

        "source_test_cells":
            int(
                len(
                    test_src
                )
            ),

        "target_test_cells":
            int(
                len(
                    test_tgt
                )
            ),

        "epochs":
            int(epochs),

        "batch_size":
            int(batch_size),
    }


    with open(
        os.path.join(
            model_out,
            "model_manifest.json"
        ),
        "w"
    ) as handle:

        json.dump(
            manifest,
            handle,
            indent=2
        )


    print(
        "\nTranslation complete."
    )

    print(
        "Forward translated morphology:",
        X_fake_drug_orig.shape
    )

    print(
        "Forward target features:",
        len(
            tgt_ae["feature_cols"]
        )
    )

    print(
        "Saved ->",
        model_out
    )


    return {

        "cell_type":
            cell_type,

        "drug":
            drug_display_name,

        "source_name":
            SRC_NAME,

        "target_name":
            TGT_NAME,

        "output_dir":
            model_out,

        "G_control_to_drug":
            G_CT,

        "G_drug_to_control":
            G_TC,

        "source_ae":
            src_ae,

        "target_ae":
            tgt_ae,

        # PRIMARY translated morphology
        "X_fake_drug_scaled":
            X_fake_drug_scaled,

        "X_fake_drug_orig":
            X_fake_drug_orig,

        "Z_fake_drug_std":
            fake_drug_std,

        "Z_fake_drug_lat":
            fake_drug_lat,

        # reverse diagnostic
        "X_fake_control_scaled":
            X_fake_control_scaled,

        "X_fake_control_orig":
            X_fake_control_orig,

        "history":
            histories,
    }

In [ ]:
control_results=[]
for cell_type in ['iPSC','Astrocytes']:
    for drug in DRUGS:
        print('CONTROL TO DRUG',cell_type,drug)
        r=run_control_to_drug_translation(cell_type,drug,ae_dir=AE_DIR,out_root=CTRL_DIR)
        control_results.append({'cell_type':cell_type,'drug':drug,'output_dir':r['output_dir']})
pd.DataFrame(control_results).to_csv(os.path.join(CTRL_DIR,'control_to_drug_summary.csv'),index=False)


## 6. Final audit — confirm all ten original-scale arrays exist


In [ ]:
from pathlib import Path
rows=[]
for drug in ALL_CONDITIONS:
    folder=Path(DEV_DIR)/drug
    arr=folder/'translated_astro_morphology_original.npy'
    feats=folder/'translated_astro_feature_names.csv'
    rows.append({'family':'development','condition':drug,'array':str(arr),'exists':arr.exists() and feats.exists()})
for cell_type in ('iPSC','Astrocytes'):
    for drug in DRUGS:
        folder=Path(CTRL_DIR)/f'{cell_type}_DMSO_to_{drug}'
        arr=folder/'X_control_to_drug_original.npy'
        feats=folder/'control_to_drug_feature_names.csv'
        rows.append({'family':'control_to_drug','condition':f'{cell_type}_{drug}','array':str(arr),'exists':arr.exists() and feats.exists()})
audit=pd.DataFrame(rows);audit.to_csv(os.path.join(EVAL_DIR,'output_audit.csv'),index=False)
display(audit)
assert audit.exists.all(), 'Some translated morphology outputs are missing: inspect table above'


## 7. Final test-set effect recovery, family recovery and drug specificity


In [ ]:
from scipy.stats import pearsonr, spearmanr

def get_real_test(condition):
    prefix=os.path.join(AE_DIR,condition)
    stats=pd.read_csv(prefix+'_scaler_stats.csv')
    X=np.load(prefix+'_X_test_scaled.npy')
    original=X*stats['scaler_scale'].to_numpy()[None,:]+stats['scaler_mean'].to_numpy()[None,:]
    return pd.DataFrame(original,columns=stats['feature'].tolist())

def load_pred_dev(drug):
    p=os.path.join(DEV_DIR,drug)
    X=np.load(os.path.join(p,'translated_astro_morphology_original.npy'))
    f=pd.read_csv(os.path.join(p,'translated_astro_feature_names.csv'))['feature'].tolist()
    return pd.DataFrame(X,columns=f)

def load_pred_ctrl(cell_type,drug):
    p=os.path.join(CTRL_DIR,f'{cell_type}_DMSO_to_{drug}')
    X=np.load(os.path.join(p,'X_control_to_drug_original.npy'))
    f=pd.read_csv(os.path.join(p,'control_to_drug_feature_names.csv'))['feature'].tolist()
    return pd.DataFrame(X,columns=f)

def morphology_family(name):
    for family in ['AreaShape','Granularity','RadialDistribution','Texture','Intensity','Correlation']:
        if family.lower() in name.lower():return family
    return 'Other'

def safe_corr(a,b,method='pearson'):
    a=np.asarray(a);b=np.asarray(b)
    ok=np.isfinite(a)&np.isfinite(b)
    if ok.sum()<3 or np.std(a[ok])<1e-12 or np.std(b[ok])<1e-12:return np.nan
    return float((pearsonr if method=='pearson' else spearmanr)(a[ok],b[ok]).statistic)

def drug_effect_table(real_control,real_drug,pred_control,pred_drug):
    features=sorted(set(real_control)&set(real_drug)&set(pred_control)&set(pred_drug))
    if len(features)<3:raise ValueError('Insufficient common named features')
    R0,R1,P0,P1=[df[features].to_numpy(dtype=float) for df in [real_control,real_drug,pred_control,pred_drug]]
    # Reference standardization frozen to measured test controls/drug, not refitted on predictions.
    sd=np.sqrt(((len(R0)-1)*np.var(R0,axis=0,ddof=1)+(len(R1)-1)*np.var(R1,axis=0,ddof=1))/max(len(R0)+len(R1)-2,1))
    valid=np.isfinite(sd)&(sd>1e-8)
    real=(R1.mean(0)-R0.mean(0))[valid]/sd[valid]
    pred=(P1.mean(0)-P0.mean(0))[valid]/sd[valid]
    return pd.DataFrame({'feature':np.array(features)[valid],'family':[morphology_family(f) for f in np.array(features)[valid]], 'real_effect':real,'pred_effect':pred, 'absolute_error':np.abs(real-pred),'sign_match':np.sign(real)==np.sign(pred)})

def effect_metrics(table, label):
    x=table['real_effect'].to_numpy();y=table['pred_effect'].to_numpy()
    return {'comparison':label,'n_features':len(table),'pearson_r':safe_corr(x,y),'spearman_r':safe_corr(x,y,'spearman'),'sign_concordance':float(np.mean(np.sign(x)==np.sign(y))),'MAE':float(np.mean(np.abs(x-y))),'RMSE':float(np.sqrt(np.mean((x-y)**2))),'no_effect_baseline_MAE':float(np.mean(np.abs(x)))}

results=[];tables={}; family_rows=[]
for drug in DRUGS:
    real_control=get_real_test('Astrocytes_DMSO')
    real_drug=get_real_test('Astrocytes_'+drug)
    pred_dev_control=load_pred_dev('DMSO')
    pred_dev_drug=load_pred_dev(drug)
    pred_ctrl_drug=load_pred_ctrl('Astrocytes',drug)
    for name,pc,pd_ in [('developmental',pred_dev_control,pred_dev_drug),('control_to_drug',real_control,pred_ctrl_drug)]:
        tab=drug_effect_table(real_control,real_drug,pc,pd_)
        label=f'{name}__{drug}'
        tables[label]=tab
        tab.to_csv(os.path.join(EVAL_DIR,f'{label}_feature_effects.csv'),index=False)
        results.append(effect_metrics(tab,label))
        for family,g in tab.groupby('family'):
            family_rows.append({'comparison':label,'family':family,**effect_metrics(g,label)})
metrics=pd.DataFrame(results)
metrics.to_csv(os.path.join(EVAL_DIR,'drug_effect_recovery_summary.csv'),index=False)
pd.DataFrame(family_rows).to_csv(os.path.join(EVAL_DIR,'morphology_family_recovery.csv'),index=False)
display(metrics)


In [ ]:
# Cross-drug specificity: align feature sets so all 9 correlations use the same features.
drug_tabs={d:tables['developmental__'+d].set_index('feature') for d in DRUGS}
common=set.intersection(*(set(t.index) for t in drug_tabs.values()))
common=sorted(common)
assert len(common)>=3, 'Too few common morphology features for cross-drug specificity'
mat=[]
for predicted_drug in DRUGS:
    row={'predicted_drug':predicted_drug}
    for measured_drug in DRUGS:
        row[measured_drug]=safe_corr(drug_tabs[predicted_drug].loc[common,'pred_effect'],drug_tabs[measured_drug].loc[common,'real_effect'])
    mat.append(row)
specificity=pd.DataFrame(mat).set_index('predicted_drug')
specificity.to_csv(os.path.join(EVAL_DIR,'cross_drug_specificity.csv'))
display(specificity)
# Rank recovery and iPSC carryover comparison on common named features.
additional=[]
for drug in DRUGS:
    t=tables['developmental__'+drug]
    idx=t.set_index('feature')
    for k in [20,50,100]:
        k=min(k,len(idx))
        top_real=set(idx['real_effect'].abs().nlargest(k).index)
        top_pred=set(idx['pred_effect'].abs().nlargest(k).index)
        additional.append({'drug':drug,'top_n':k,'jaccard':len(top_real&top_pred)/len(top_real|top_pred),'overlap':len(top_real&top_pred)})
    i0=get_real_test('iPSC_DMSO');i1=get_real_test('iPSC_'+drug)
    common=sorted(set(idx.index)&set(i0)&set(i1))
    if len(common)>=3:
        carry=(i1[common].mean()-i0[common].mean()).to_numpy()
        # Rescale all three signatures using observed astrocyte test SD for comparable feature units
        a0=get_real_test('Astrocytes_DMSO')[common].to_numpy();a1=get_real_test('Astrocytes_'+drug)[common].to_numpy()
        sd=np.sqrt(((len(a0)-1)*a0.var(0,ddof=1)+(len(a1)-1)*a1.var(0,ddof=1))/(len(a0)+len(a1)-2))
        valid=sd>1e-8
        carry=carry[valid]/sd[valid]
        measured=idx.loc[common,'real_effect'].to_numpy()[valid]
        predicted=idx.loc[common,'pred_effect'].to_numpy()[valid]
        additional.append({'drug':drug,'analysis':'iPSC_carryover','n_features':len(measured),'carryover_pearson':safe_corr(measured,carry),'model_pearson':safe_corr(measured,predicted),'carryover_MAE':float(np.mean(np.abs(measured-carry))),'model_MAE_same_features':float(np.mean(np.abs(measured-predicted)))})
pd.DataFrame(additional).to_csv(os.path.join(EVAL_DIR,'biological_interpretation_additional.csv'),index=False)
display(pd.DataFrame(additional))
